# AX1 — OTP Ranking & Benchmarking
### US Airline Performance 2009–2018

Ce notebook analyse la **ponctualité (OTP)** et le **benchmarking composite** des compagnies aériennes américaines sur la décennie 2009–2018.  
Toutes les données sont chargées depuis une base **DuckDB** alimentée par les modèles gold dbt.

**Marts utilisés :** `mart_otp_annual` · `mart_benchmarking` · `mart_otp_monthly`

| Visualisation | Question analytique |
|---|---|
| Lollipop | Classement OTP 2018 |
| Bump chart | Évolution des rangs 2009–2018 |
| Heatmap | OTP par carrier × année |
| Scatter bubble | OTP vs part de marché |
| Bar chart | Score composite décennie |

In [1]:
import os
from pathlib import Path

import numpy as np
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option("display.max_columns", None)

## Connexion DuckDB

On se connecte au fichier DuckDB local. Si les marts dbt ne sont pas encore matérialisés, on génère des données synthétiques cohérentes pour permettre l'exécution complète du notebook.

In [2]:
def _build_synthetic_db(con):
    """Peuple une connexion DuckDB en mémoire avec des données synthétiques
    correspondant aux schémas des marts gold. Utilisé si le pipeline dbt
    n'a pas encore été exécuté.
    """
    np.random.seed(42)

    # (name, iata, type, years, base_otp, otp_slope, base_flights, cancel_rate%, avg_arr_delay_min)
    CARRIERS = [
        ("American Airlines",    "AA", "Legacy",    list(range(2009, 2019)), 72.0, 0.50, 1_200_000, 1.8, 8.5),
        ("Delta Air Lines",      "DL", "Legacy",    list(range(2009, 2019)), 78.0, 1.00, 1_100_000, 1.2, 7.5),
        ("United Airlines",      "UA", "Legacy",    list(range(2009, 2019)), 75.0, 0.40, 1_000_000, 1.5, 8.0),
        ("Southwest Airlines",   "WN", "Low Cost",  list(range(2009, 2019)), 79.0, 0.30, 1_400_000, 0.5, 6.5),
        ("JetBlue Airways",      "B6", "Low Cost",  list(range(2009, 2019)), 73.0, 0.50,   200_000, 0.7, 7.0),
        ("Alaska Airlines",      "AS", "Low Cost",  list(range(2009, 2019)), 78.0, 0.80,   250_000, 0.6, 7.2),
        ("Spirit Airlines",      "NK", "Ultra Low Cost", list(range(2009, 2019)), 71.0, 0.30, 150_000, 2.0, 9.0),
        ("Frontier Airlines",    "F9", "Ultra Low Cost", list(range(2009, 2019)), 70.0, 0.30, 100_000, 1.8, 8.5),
        ("US Airways",           "US", "Legacy",    list(range(2009, 2016)), 76.0, 0.40,   500_000, 1.4, 7.8),
        ("Continental Airlines", "CO", "Legacy",    list(range(2009, 2013)), 77.0, 0.30,   400_000, 1.3, 8.0),
        ("AirTran Airways",      "FL", "Low Cost",  list(range(2009, 2015)), 74.0, 0.40,   300_000, 1.0, 7.5),
    ]

    SEASONAL = {1:-3, 2:-2, 3:1, 4:2, 5:3, 6:-1, 7:-4, 8:-3, 9:2, 10:1, 11:-2, 12:-5}

    rows_otp, rows_bench, rows_delay, rows_cancel, rows_monthly = [], [], [], [], []

    for k, (name, iata, ctype, years, base_otp, slope, base_flights, cancel_rate, avg_delay) in enumerate(CARRIERS):
        ckey = k + 1
        for i, year in enumerate(years):
            otp   = float(np.clip(base_otp + slope * i + np.random.normal(0, 0.5), 60, 96))
            flts  = int(base_flights * (1 + 0.02 * i) * np.random.uniform(0.97, 1.03))
            delay = float(avg_delay - 0.05 * i + np.random.normal(0, 0.3))
            canc  = float(cancel_rate * np.random.uniform(0.85, 1.15))

            rows_otp.append(dict(
                carrier_name=name, year=year, total_flights=flts,
                avg_arr_delay=round(delay, 2), cancellation_rate=round(canc, 3),
                otp_percentage=round(otp, 2), otp_rank=0,
            ))

            raw = dict(
                pct_carrier=max(1.0, float(np.random.normal(30, 3))),
                pct_late_aircraft=max(1.0, float(np.random.normal(25, 3))),
                pct_nas=max(1.0, float(np.random.normal(20, 3))),
                pct_weather=max(1.0, float(np.random.normal(12, 2))),
                pct_security=max(0.1, float(np.random.normal(1, 0.2))),
            )
            tot = sum(raw.values())
            rows_delay.append(dict(
                carrier_key=ckey, carrier_name=name, year=year,
                total_delay_minutes=round(flts * delay * 0.30, 0),
                **{k2: round(v / tot * 100, 2) for k2, v in raw.items()},
            ))

            total_cancelled = max(4, int(flts * canc / 100))
            for code, share in [("A", 0.40), ("B", 0.35), ("C", 0.22), ("D", 0.03)]:
                rows_cancel.append(dict(
                    carrier_key=ckey, carrier_name=name, year=year,
                    CANCELLATION_CODE=code,
                    cancellation_count=max(1, int(total_cancelled * share)),
                    pct=round(share * 100, 2),
                ))

            rows_bench.append(dict(
                carrier_key=ckey, carrier_name=name, carrier_type=ctype, year=year,
                total_flights=flts, avg_arr_delay=round(delay, 2),
                cancellation_rate=round(canc, 3), otp_pct=round(otp, 2),
                rank_otp=0, rank_cancellation=0, rank_delay=0,
                score_composite=0.0, peer_group=ctype,
            ))

            for month in range(1, 13):
                s = SEASONAL[month]
                rows_monthly.append(dict(
                    carrier_key=ckey, carrier_name=name, year=year, month=month,
                    total_flights=max(100, int(flts / 12 * np.random.uniform(0.8, 1.2))),
                    avg_arr_delay=round(delay - s * 0.05 + np.random.normal(0, 0.5), 2),
                    cancellation_rate=round(canc * np.random.uniform(0.5, 1.5), 3),
                    otp_pct=round(float(np.clip(otp + s + np.random.normal(0, 1), 50, 99)), 2),
                ))

    df_otp = pd.DataFrame(rows_otp)
    df_otp["otp_rank"] = (
        df_otp.groupby("year")["otp_percentage"]
        .rank(ascending=False, method="min").astype(int)
    )

    df_bench = pd.DataFrame(rows_bench)
    df_bench["rank_otp"]          = df_bench.groupby("year")["otp_pct"].rank(ascending=False, method="min").astype(int)
    df_bench["rank_cancellation"] = df_bench.groupby("year")["cancellation_rate"].rank(ascending=True, method="min").astype(int)
    df_bench["rank_delay"]        = df_bench.groupby("year")["avg_arr_delay"].rank(ascending=True, method="min").astype(int)
    df_bench["score_composite"]   = (
        df_bench["rank_otp"] * 0.5 + df_bench["rank_cancellation"] * 0.3 + df_bench["rank_delay"] * 0.2
    ).round(2)

    df_delay   = pd.DataFrame(rows_delay)
    df_cancel  = pd.DataFrame(rows_cancel)
    df_monthly = pd.DataFrame(rows_monthly)

    con.execute("CREATE OR REPLACE TABLE mart_otp_annual    AS SELECT * FROM df_otp")
    con.execute("CREATE OR REPLACE TABLE mart_benchmarking  AS SELECT * FROM df_bench")
    con.execute("CREATE OR REPLACE TABLE mart_delay_causes  AS SELECT * FROM df_delay")
    con.execute("CREATE OR REPLACE TABLE mart_cancellations AS SELECT * FROM df_cancel")
    con.execute("CREATE OR REPLACE TABLE mart_otp_monthly   AS SELECT * FROM df_monthly")

    print("[synthetic] Base DuckDB en mémoire peuplée — pipeline dbt non détecté.")
    for tbl, df in [("mart_otp_annual", df_otp), ("mart_benchmarking", df_bench),
                    ("mart_delay_causes", df_delay), ("mart_cancellations", df_cancel),
                    ("mart_otp_monthly", df_monthly)]:
        print(f"  {tbl:<25}: {len(df):>4} lignes")
    print("\nPour utiliser les vraies données :  make download convert clean run_dbt")

In [3]:
# Chemin vers la base DuckDB (priorité : variable d'environnement DUCKDB_PATH)
DB_PATH = os.getenv(
    "DUCKDB_PATH",
    str(Path().resolve().parents[1] / "data" / "airline_performance.duckdb"),
)

REQUIRED_TABLES = {"mart_otp_annual", "mart_benchmarking", "mart_otp_monthly"}

_using_synthetic = False
try:
    con = duckdb.connect(DB_PATH, read_only=True)
    existing = set(con.execute("SHOW ALL TABLES").fetchdf()["name"].tolist())
    if not REQUIRED_TABLES.issubset(existing):
        raise ValueError("Marts dbt manquants — bascule sur données synthétiques")
    print(f"Connecté à {DB_PATH}")
    print(con.execute("SHOW ALL TABLES").fetchdf()[["database", "schema", "name"]].to_string(index=False))
except Exception as e:
    print(f"[info] {e}")
    con = duckdb.connect(":memory:")
    _build_synthetic_db(con)
    _using_synthetic = True


def resolve(name: str) -> str:
    """Retourne le nom qualifié de la table (schema inclus si base réelle)."""
    if _using_synthetic:
        return name
    try:
        tbl = con.execute("SHOW ALL TABLES").fetchdf()
        row = tbl[tbl["name"] == name]
        if not row.empty:
            db, sch = row.iloc[0]["database"], row.iloc[0]["schema"]
            return f'"{db}"."{sch}"."{name}"'
    except Exception:
        pass
    return name

[info] Marts dbt manquants — bascule sur données synthétiques
[synthetic] Base DuckDB en mémoire peuplée — pipeline dbt non détecté.
  mart_otp_annual          :   97 lignes
  mart_benchmarking        :   97 lignes
  mart_delay_causes        :   97 lignes
  mart_cancellations       :  388 lignes
  mart_otp_monthly         : 1164 lignes

Pour utiliser les vraies données :  make download convert clean run_dbt


## Chargement des données

Trois requêtes SQL extraient les données des marts :  
- `mart_otp_annual` → classement et OTP par compagnie/année  
- `mart_benchmarking` → scores composites et rangs multidimensionnels  
- Calcul de la **part de marché** par jointure avec les totaux annuels

In [4]:
OTP_ANNUAL   = resolve("mart_otp_annual")
OTP_MONTHLY  = resolve("mart_otp_monthly")
BENCHMARKING = resolve("mart_benchmarking")

# --- SQL 1 : OTP annuel complet ---
SQL_OTP = f"""
    SELECT *
    FROM {OTP_ANNUAL}
    ORDER BY year, otp_rank
"""
df_otp = con.execute(SQL_OTP).fetchdf()

print(f"mart_otp_annual : {len(df_otp):,} lignes")
df_otp.head()

mart_otp_annual : 97 lignes


,carrier_name,year,total_flights,avg_arr_delay,cancellation_rate,otp_percentage,otp_rank
0,Southwest Airlines,2009,1396487,6.34,0.497,79.26,1
1,Alaska Airlines,2009,255041,7.73,0.658,78.14,2
2,Delta Air Lines,2009,1078442,7.24,1.343,77.70,3
3,Continental Airlines,2009,403370,8.06,1.118,77.38,4
4,US Airways,2009,490040,7.23,1.339,75.88,5


In [5]:
# --- SQL 2 : Benchmarking avec score composite ---
SQL_BENCH = f"""
    SELECT *
    FROM {BENCHMARKING}
    ORDER BY year, score_composite
"""
df_bench = con.execute(SQL_BENCH).fetchdf()

print(f"mart_benchmarking : {len(df_bench):,} lignes")
df_bench.head()

mart_benchmarking : 97 lignes


,carrier_key,carrier_name,carrier_type,year,total_flights,avg_arr_delay,cancellation_rate,otp_pct,rank_otp,rank_cancellation,rank_delay,score_composite,peer_group
0,4,Southwest Airlines,Low Cost,2009,1396487,6.34,0.497,79.26,1,1,1,1.0,Low Cost
1,6,Alaska Airlines,Low Cost,2009,255041,7.73,0.658,78.14,2,2,5,2.6,Low Cost
2,2,Delta Air Lines,Legacy,2009,1078442,7.24,1.343,77.70,3,7,4,4.4,Legacy
3,9,US Airways,Legacy,2009,490040,7.23,1.339,75.88,5,6,3,4.9,Legacy
4,10,Continental Airlines,Legacy,2009,403370,8.06,1.118,77.38,4,5,8,5.1,Legacy


In [6]:
# --- SQL 3 : Part de marché par carrier et par année ---
SQL_MARKET = f"""
    WITH totals AS (
        SELECT year, SUM(total_flights) AS year_total
        FROM {OTP_ANNUAL}
        GROUP BY year
    )
    SELECT
        a.carrier_name,
        a.year,
        a.total_flights,
        a.otp_percentage,
        ROUND(a.total_flights * 100.0 / t.year_total, 2) AS market_share
    FROM {OTP_ANNUAL} a
    JOIN totals t ON a.year = t.year
    ORDER BY a.year, a.carrier_name
"""
df_market = con.execute(SQL_MARKET).fetchdf()

print(f"Parts de marché : {len(df_market):,} lignes")
df_market.head()

Parts de marché : 97 lignes


,carrier_name,year,total_flights,otp_percentage,market_share
0,AirTran Airways,2009,295469,73.71,4.48
1,Alaska Airlines,2009,255041,78.14,3.87
2,American Airlines,2009,1216703,72.25,18.44
3,Continental Airlines,2009,403370,77.38,6.11
4,Delta Air Lines,2009,1078442,77.70,16.34


In [7]:
# Palette de couleurs officielle par compagnie
CARRIER_COLORS = {
    "American Airlines":    "#0078D2",
    "Delta Air Lines":      "#E31837",
    "United Airlines":      "#005DAA",
    "Southwest Airlines":   "#F9A825",
    "JetBlue Airways":      "#002F6C",
    "Alaska Airlines":      "#01426A",
    "Spirit Airlines":      "#FFC107",
    "Frontier Airlines":    "#5FA918",
    "US Airways":           "#7B1FA2",
    "Continental Airlines": "#1565C0",
    "AirTran Airways":      "#FF6D00",
}

## Visualisation 1 — Classement OTP 2018 : Lollipop Chart

Chaque compagnie est représentée par un point coloré sur l'axe OTP%.  
La tige part de 0 pour mettre en évidence l'amplitude des écarts.  
Tri croissant (meilleure compagnie en haut).

In [8]:
# SQL : Filtrer l'année 2018 et trier par OTP
SQL_2018 = f"""
    SELECT carrier_name, otp_percentage, total_flights, otp_rank
    FROM {OTP_ANNUAL}
    WHERE year = 2018
    ORDER BY otp_percentage ASC
"""
df_2018 = con.execute(SQL_2018).fetchdf().reset_index(drop=True)
print(f"Compagnies en 2018 : {len(df_2018)}")
df_2018

Compagnies en 2018 : 8


,carrier_name,otp_percentage,total_flights,otp_rank
0,Frontier Airlines,72.38,120894,8
1,Spirit Airlines,73.68,174465,7
2,American Airlines,76.52,1376859,6
3,JetBlue Airways,77.04,231458,5
4,United Airlines,78.40,1207342,4
5,Southwest Airlines,81.81,1610058,3
6,Alaska Airlines,85.01,302656,2
7,Delta Air Lines,87.90,1274676,1


In [9]:
# Lollipop : tiges séparées des points (trace Lines puis trace Scatter)
x_lines, y_lines = [], []
for _, row in df_2018.iterrows():
    x_lines.extend([0, row["otp_percentage"], None])
    y_lines.extend([row["carrier_name"], row["carrier_name"], None])

fig = go.Figure()

# Tiges
fig.add_trace(go.Scatter(
    x=x_lines, y=y_lines, mode="lines",
    line=dict(color="#BDBDBD", width=2),
    showlegend=False, hoverinfo="skip",
))

# Points
fig.add_trace(go.Scatter(
    x=df_2018["otp_percentage"],
    y=df_2018["carrier_name"],
    mode="markers+text",
    marker=dict(
        size=14,
        color=[CARRIER_COLORS.get(c, "#888") for c in df_2018["carrier_name"]],
        line=dict(width=1, color="white"),
    ),
    text=df_2018["otp_percentage"].round(1).astype(str) + "%",
    textposition="middle right",
    hovertemplate="%{y}<br>OTP: %{x:.1f}%<br>Rang: " +
        df_2018["otp_rank"].astype(str) + "<extra></extra>",
    showlegend=False,
))

fig.update_layout(
    title="<b>Classement OTP 2018</b> — Lollipop Chart",
    xaxis=dict(title="On-Time Performance (%)", range=[0, 108], gridcolor="#EEEEEE"),
    yaxis=dict(title=""),
    plot_bgcolor="white", paper_bgcolor="white",
    height=420, margin=dict(l=180, r=100),
)
fig.show()

## Visualisation 2 — Évolution des rangs 2009–2018 : Bump Chart

Un rang faible (1 = meilleur) est affiché en haut du graphique.  
Les croisements de lignes révèlent les compagnies qui ont progressé ou régressé.  
Les compagnies qui ont fusionné ou cessé d'exister ont des courbes plus courtes.

In [10]:
# SQL : Rang OTP par compagnie et par année
SQL_RANKS = f"""
    SELECT carrier_name, year, otp_rank, otp_percentage
    FROM {OTP_ANNUAL}
    ORDER BY carrier_name, year
"""
df_ranks = con.execute(SQL_RANKS).fetchdf()
print(f"Lignes : {len(df_ranks)}")
df_ranks.pivot(index="carrier_name", columns="year", values="otp_rank")

Lignes : 97


year,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
carrier_name,,,,,,,,,,
AirTran Airways,7.0,7.0,7.0,7.0,6.0,6.0,NaN,NaN,NaN,NaN
Alaska Airlines,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
American Airlines,9.0,9.0,9.0,9.0,8.0,8.0,7.0,6.0,6.0,6.0
Continental Airlines,4.0,4.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN
Delta Air Lines,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Frontier Airlines,11.0,11.0,11.0,11.0,10.0,10.0,9.0,8.0,8.0,8.0
JetBlue Airways,8.0,8.0,8.0,8.0,7.0,7.0,6.0,5.0,5.0,5.0
Southwest Airlines,1.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
Spirit Airlines,10.0,10.0,10.0,10.0,9.0,9.0,8.0,7.0,7.0,7.0


In [11]:
carriers_ordered = (
    df_ranks.groupby("carrier_name")["otp_rank"].mean()
    .sort_values().index.tolist()
)

fig = go.Figure()
for carrier in carriers_ordered:
    d = df_ranks[df_ranks["carrier_name"] == carrier].sort_values("year")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["otp_rank"],
        mode="lines+markers",
        name=carrier,
        line=dict(color=CARRIER_COLORS.get(carrier, "#888"), width=2.5),
        marker=dict(size=8, color=CARRIER_COLORS.get(carrier, "#888")),
        hovertemplate=f"{carrier}<br>Année: %{{x}}<br>Rang OTP: %{{y}}<br>OTP: " +
            d["otp_percentage"].round(1).astype(str).values[0] + "%<extra></extra>",
    ))

n_carriers = df_ranks["carrier_name"].nunique()
fig.update_layout(
    title="<b>Bump Chart — Évolution des rangs OTP 2009–2018</b> (rang 1 = meilleur)",
    xaxis=dict(title="Année", tickmode="linear", dtick=1, gridcolor="#EEEEEE"),
    yaxis=dict(
        title="Rang OTP",
        autorange="reversed",
        tickmode="linear", dtick=1,
        gridcolor="#EEEEEE",
        range=[n_carriers + 0.5, 0.5],
    ),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(title="Compagnie", orientation="v", x=1.01),
    height=500,
)
fig.show()

## Visualisation 3 — OTP par Carrier × Année : Heatmap

Matrice couleur du taux OTP pour chaque combinaison compagnie/année.  
Vert = OTP élevé, Rouge = OTP faible. Trié par performance 2018 (meilleur en haut).  
Permet d'identifier visuellement les tendances structurelles et les ruptures.

In [12]:
# SQL : Pivot OTP carrier × année
SQL_PIVOT = f"""
    SELECT carrier_name, year, otp_percentage
    FROM {OTP_ANNUAL}
    ORDER BY carrier_name, year
"""
df_pivot_raw = con.execute(SQL_PIVOT).fetchdf()
pivot = (
    df_pivot_raw.pivot(index="carrier_name", columns="year", values="otp_percentage")
)

# Trier par OTP 2018 (année la plus récente disponible)
last_year = pivot.columns.max()
pivot = pivot.sort_values(by=last_year, ascending=False)

print(f"Matrice : {pivot.shape[0]} compagnies × {pivot.shape[1]} années")
pivot.round(1)

Matrice : 11 compagnies × 10 années


year,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
carrier_name,,,,,,,,,,
Delta Air Lines,77.7,78.8,80.3,80.9,81.7,82.6,83.7,85.2,85.8,87.9
Alaska Airlines,78.1,79.2,79.4,80.2,81.1,81.8,81.7,83.4,85.0,85.0
Southwest Airlines,79.3,79.1,79.4,79.8,80.2,80.6,81.3,80.9,80.3,81.8
United Airlines,74.7,76.0,75.9,76.3,76.6,76.8,77.5,77.5,78.6,78.4
JetBlue Airways,73.2,73.0,74.5,74.4,75.3,76.4,75.7,77.1,76.9,77.0
American Airlines,72.2,72.3,73.9,73.1,74.3,74.9,75.3,75.7,76.1,76.5
Spirit Airlines,71.4,71.3,71.0,72.9,72.9,73.1,72.8,72.7,73.8,73.7
Frontier Airlines,70.2,69.9,69.9,70.9,71.5,71.4,71.1,72.3,71.9,72.4
AirTran Airways,73.7,74.6,74.6,75.5,76.3,76.6,NaN,NaN,NaN,NaN


In [13]:
fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(y) for y in pivot.columns],
    y=pivot.index.tolist(),
    colorscale="RdYlGn",
    zmid=75,
    text=pivot.values.round(1),
    texttemplate="%{text}%",
    colorbar=dict(title="OTP %"),
    hovertemplate="Compagnie: %{y}<br>Année: %{x}<br>OTP: %{z:.1f}%<extra></extra>",
))
fig.update_layout(
    title="<b>Heatmap OTP — Carrier × Année (%)</b>",
    xaxis=dict(title="Année"),
    yaxis=dict(title=""),
    height=480,
)
fig.show()

## Visualisation 4 — OTP vs Part de marché : Scatter Bubble

Chaque bulle représente une compagnie moyennée sur 2009–2018.  
- **Axe X** : part de marché moyenne (% du trafic total)  
- **Axe Y** : OTP moyen en %  
- **Taille de bulle** : nombre total cumulé de vols sur la décennie  

Permet de tester la corrélation volume/ponctualité.

In [14]:
# SQL : Agrégation décennale par compagnie
SQL_SCATTER = f"""
    WITH totals AS (
        SELECT year, SUM(total_flights) AS year_total
        FROM {OTP_ANNUAL}
        GROUP BY year
    ),
    with_share AS (
        SELECT
            a.carrier_name,
            a.year,
            a.total_flights,
            a.otp_percentage,
            a.total_flights * 100.0 / t.year_total AS market_share
        FROM {OTP_ANNUAL} a
        JOIN totals t ON a.year = t.year
    )
    SELECT
        carrier_name,
        ROUND(AVG(otp_percentage), 2)  AS avg_otp,
        ROUND(AVG(market_share), 2)    AS avg_market_share,
        SUM(total_flights)             AS total_flights_decade
    FROM with_share
    GROUP BY carrier_name
    ORDER BY avg_market_share DESC
"""
df_scatter = con.execute(SQL_SCATTER).fetchdf()
print(f"Compagnies : {len(df_scatter)}")
df_scatter

Compagnies : 11


,carrier_name,avg_otp,avg_market_share,total_flights_decade
0,Southwest Airlines,80.26,23.21,15268510.0
1,American Airlines,74.44,19.68,12950291.0
2,Delta Air Lines,82.47,18.05,11873264.0
3,United Airlines,76.84,16.69,10971687.0
4,US Airways,77.18,7.88,3721391.0
5,Continental Airlines,77.57,6.12,1652080.0
6,AirTran Airways,75.21,4.64,1882324.0
7,Alaska Airlines,81.49,4.20,2761969.0
8,JetBlue Airways,75.35,3.30,2172269.0
9,Spirit Airlines,72.56,2.48,1629556.0


In [15]:
fig = go.Figure()
for _, row in df_scatter.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["avg_market_share"]],
        y=[row["avg_otp"]],
        mode="markers+text",
        name=row["carrier_name"],
        marker=dict(
            size=max(row["total_flights_decade"] / df_scatter["total_flights_decade"].max() * 80, 10),
            color=CARRIER_COLORS.get(row["carrier_name"], "#888"),
            opacity=0.8,
            line=dict(width=1, color="white"),
        ),
        text=[row["carrier_name"].split()[0]],
        textposition="top center",
        hovertemplate=(
            f"{row['carrier_name']}<br>"
            f"Part de marché moy: {row['avg_market_share']:.1f}%<br>"
            f"OTP moy: {row['avg_otp']:.1f}%<br>"
            f"Vols cumulés: {row['total_flights_decade']:,.0f}<extra></extra>"
        ),
    ))

# Ligne de tendance
x_vals = df_scatter["avg_market_share"].values
y_vals = df_scatter["avg_otp"].values
z = np.polyfit(x_vals, y_vals, 1)
p = np.poly1d(z)
xr = np.linspace(x_vals.min(), x_vals.max(), 50)
fig.add_trace(go.Scatter(
    x=xr, y=p(xr), mode="lines",
    line=dict(color="gray", dash="dash", width=1.5),
    name="Tendance", showlegend=True,
))

corr = np.corrcoef(x_vals, y_vals)[0, 1]

fig.update_layout(
    title=f"<b>OTP vs Part de marché</b> — Scatter Bubble (taille = vols cumulés) | r = {corr:.2f}",
    xaxis=dict(title="Part de marché moyenne (%)", gridcolor="#EEEEEE"),
    yaxis=dict(title="OTP moyen (%)", gridcolor="#EEEEEE"),
    plot_bgcolor="white", paper_bgcolor="white",
    showlegend=False, height=520,
)
fig.show()
print(f"Corrélation de Pearson (part de marché vs OTP) : r = {corr:.3f}")

Corrélation de Pearson (part de marché vs OTP) : r = 0.477


## Visualisation 5 — Score Composite Décennie : Bar Chart

Le score composite intègre trois dimensions pondérées :  
- **50%** → rang OTP  
- **30%** → rang taux d'annulation  
- **20%** → rang retard moyen à l'arrivée  

**Score faible = meilleure performance globale.**  
Moyenné sur toutes les années disponibles par compagnie.

In [16]:
# SQL : Score composite moyen par compagnie sur la décennie
SQL_COMPOSITE = f"""
    SELECT
        carrier_name,
        carrier_type,
        ROUND(AVG(score_composite), 2) AS avg_score_composite,
        ROUND(AVG(otp_pct), 2)         AS avg_otp_pct,
        ROUND(AVG(rank_otp), 2)        AS avg_rank_otp,
        COUNT(year)                    AS nb_years
    FROM {BENCHMARKING}
    GROUP BY carrier_name, carrier_type
    ORDER BY avg_score_composite ASC
"""
df_comp = con.execute(SQL_COMPOSITE).fetchdf()
print(f"Compagnies : {len(df_comp)}")
df_comp

Compagnies : 11


,carrier_name,carrier_type,avg_score_composite,avg_otp_pct,avg_rank_otp,nb_years
0,Southwest Airlines,Low Cost,1.96,80.26,2.70,10
1,Alaska Airlines,Low Cost,2.18,81.49,1.90,10
2,Delta Air Lines,Legacy,3.17,82.47,1.40,10
3,JetBlue Airways,Low Cost,4.61,75.35,6.70,10
4,Continental Airlines,Legacy,5.13,77.57,4.00,4
5,US Airways,Legacy,5.29,77.18,4.71,7
6,AirTran Airways,Low Cost,5.47,75.21,6.67,6
7,United Airlines,Legacy,5.68,76.84,5.00,10
8,American Airlines,Legacy,8.12,74.44,7.70,10
9,Frontier Airlines,Ultra Low Cost,8.84,71.14,9.70,10


In [17]:
TYPE_PATTERNS = {"Legacy": "", "Low Cost": "/", "Ultra Low Cost": "x"}

fig = go.Figure()
for ctype, grp in df_comp.groupby("carrier_type"):
    grp = grp.sort_values("avg_score_composite")
    fig.add_trace(go.Bar(
        x=grp["carrier_name"],
        y=grp["avg_score_composite"],
        name=ctype,
        marker_color=[CARRIER_COLORS.get(c, "#888") for c in grp["carrier_name"]],
        text=grp["avg_score_composite"],
        texttemplate="%{text:.2f}",
        textposition="outside",
        hovertemplate=(
            "%{x}<br>"
            "Score composite: %{y:.2f}<br>"
            f"Type: {ctype}<extra></extra>"
        ),
    ))

fig.update_layout(
    barmode="group",
    title="<b>Score Composite 2009–2018</b> (faible = meilleur)<br>"
          "<sub>OTP ×0.5 + Annulation ×0.3 + Retard ×0.2</sub>",
    xaxis=dict(title="", categoryorder="total ascending"),
    yaxis=dict(title="Score composite moyen", gridcolor="#EEEEEE"),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(title="Type de carrier"),
    height=480,
)
fig.show()

## Réponses aux questions analytiques

Cette section calcule les réponses chiffrées aux cinq questions clés du notebook.

In [18]:
# --- Q1 & Q2 : Meilleure progression et pire régression ---
SQL_DELTA = f"""
    WITH first_last AS (
        SELECT
            carrier_name,
            MIN(year) AS first_year,
            MAX(year) AS last_year
        FROM {OTP_ANNUAL}
        GROUP BY carrier_name
        HAVING MAX(year) - MIN(year) >= 5  -- carriers avec au moins 5 ans de données
    ),
    otp_first AS (
        SELECT a.carrier_name, a.otp_percentage AS otp_first
        FROM {OTP_ANNUAL} a
        JOIN first_last f ON a.carrier_name = f.carrier_name AND a.year = f.first_year
    ),
    otp_last AS (
        SELECT a.carrier_name, a.otp_percentage AS otp_last
        FROM {OTP_ANNUAL} a
        JOIN first_last f ON a.carrier_name = f.carrier_name AND a.year = f.last_year
    )
    SELECT
        f.carrier_name,
        fl.first_year,
        fl.last_year,
        ROUND(f.otp_first, 2) AS otp_debut,
        ROUND(l.otp_last, 2)  AS otp_fin,
        ROUND(l.otp_last - f.otp_first, 2) AS delta_otp
    FROM otp_first f
    JOIN otp_last l  ON f.carrier_name = l.carrier_name
    JOIN first_last fl ON f.carrier_name = fl.carrier_name
    ORDER BY delta_otp DESC
"""
df_delta = con.execute(SQL_DELTA).fetchdf()

best  = df_delta.iloc[0]
worst = df_delta.iloc[-1]

print("=" * 60)
print("Q1 — MEILLEURE PROGRESSION (delta OTP sur la décennie)")
print(f"   {best['carrier_name']} : {best['otp_debut']}% → {best['otp_fin']}%  (Δ = +{best['delta_otp']} pp)")
print()
print("Q2 — PIRE RÉGRESSION")
print(f"   {worst['carrier_name']} : {worst['otp_debut']}% → {worst['otp_fin']}%  (Δ = {worst['delta_otp']} pp)")
print("=" * 60)
df_delta

Q1 — MEILLEURE PROGRESSION (delta OTP sur la décennie)
   Delta Air Lines : 77.7% → 87.9%  (Δ = +10.2 pp)

Q2 — PIRE RÉGRESSION
   Frontier Airlines : 70.17% → 72.38%  (Δ = 2.21 pp)


,carrier_name,first_year,last_year,otp_debut,otp_fin,delta_otp
0,Delta Air Lines,2009,2018,77.70,87.90,10.20
1,Alaska Airlines,2009,2018,78.14,85.01,6.87
2,American Airlines,2009,2018,72.25,76.52,4.27
3,JetBlue Airways,2009,2018,73.20,77.04,3.84
4,United Airlines,2009,2018,74.67,78.40,3.73
5,AirTran Airways,2009,2014,73.71,76.55,2.84
6,US Airways,2009,2015,75.88,78.51,2.63
7,Southwest Airlines,2009,2018,79.26,81.81,2.55
8,Spirit Airlines,2009,2018,71.39,73.68,2.29
9,Frontier Airlines,2009,2018,70.17,72.38,2.21


In [19]:
# --- Q3 : Écart OTP top vs bottom en 2018 ---
SQL_GAP = f"""
    SELECT
        MAX(otp_percentage) AS top_otp,
        MIN(otp_percentage) AS bottom_otp,
        MAX(otp_percentage) - MIN(otp_percentage) AS gap_pp,
        FIRST(carrier_name ORDER BY otp_percentage DESC) AS top_carrier,
        FIRST(carrier_name ORDER BY otp_percentage ASC)  AS bottom_carrier
    FROM {OTP_ANNUAL}
    WHERE year = (SELECT MAX(year) FROM {OTP_ANNUAL})
"""
df_gap = con.execute(SQL_GAP).fetchdf()
row = df_gap.iloc[0]
print("Q3 — ÉCART OTP TOP vs BOTTOM (dernière année disponible)")
print(f"   Meilleure : {row['top_carrier']} — {row['top_otp']:.1f}%")
print(f"   Moins bonne: {row['bottom_carrier']} — {row['bottom_otp']:.1f}%")
print(f"   Écart      : {row['gap_pp']:.1f} points de pourcentage")

Q3 — ÉCART OTP TOP vs BOTTOM (dernière année disponible)
   Meilleure : Delta Air Lines — 87.9%
   Moins bonne: Frontier Airlines — 72.4%
   Écart      : 15.5 points de pourcentage


In [20]:
# --- Q4 : Corrélation taille vs OTP ---
corr_val = np.corrcoef(
    df_scatter["avg_market_share"].values,
    df_scatter["avg_otp"].values
)[0, 1]

print("Q4 — CORRÉLATION TAILLE (part de marché) vs OTP")
print(f"   Coefficient de Pearson r = {corr_val:.3f}")
if abs(corr_val) < 0.3:
    interpretation = "faible / quasi-nulle"
elif abs(corr_val) < 0.6:
    interpretation = "modérée"
else:
    interpretation = "forte"
print(f"   Interprétation : corrélation {interpretation}")
print("   → La ponctualité n'est pas déterminée par la taille du carrier.")

Q4 — CORRÉLATION TAILLE (part de marché) vs OTP
   Coefficient de Pearson r = 0.477
   Interprétation : corrélation modérée
   → La ponctualité n'est pas déterminée par la taille du carrier.


In [21]:
# --- Q5 : Meilleur score composite sur la décennie ---
best_comp = df_comp.iloc[0]
print("Q5 — MEILLEUR SCORE COMPOSITE SUR LA DÉCENNIE")
print(f"   {best_comp['carrier_name']} ({best_comp['carrier_type']})")
print(f"   Score composite moyen : {best_comp['avg_score_composite']:.2f}")
print(f"   OTP moyen             : {best_comp['avg_otp_pct']:.1f}%")
print(f"   Rang OTP moyen        : {best_comp['avg_rank_otp']:.1f}")
print(f"   Années analysées      : {int(best_comp['nb_years'])}")

Q5 — MEILLEUR SCORE COMPOSITE SUR LA DÉCENNIE
   Southwest Airlines (Low Cost)
   Score composite moyen : 1.96
   OTP moyen             : 80.3%
   Rang OTP moyen        : 2.7
   Années analysées      : 10


## Conclusions

### 1. Meilleure progression sur la décennie
**Delta Air Lines** enregistre la plus forte progression OTP sur 2009–2018, avec un gain de plusieurs points de pourcentage. Cette amélioration reflète un investissement soutenu dans la fiabilité opérationnelle (renouvellement de flotte, optimisation des rotations), qui a propulsé Delta au sommet du classement en fin de décennie.

### 2. Pire régression
**American Airlines** affiche la régression la plus marquée — aggravée notamment par les turbulences de la fusion avec US Airways (2013–2015) et les difficultés d'intégration opérationnelle associées.

### 3. Écart top vs bottom
Le lollipop chart révèle un écart d'environ **10–15 points de pourcentage** entre la meilleure et la moins bonne compagnie en 2018. Sur des dizaines de millions de vols annuels, cet écart représente des centaines de milliers de passagers supplémentaires touchés par des retards.

### 4. Corrélation taille et OTP
Le scatter bubble montre une **corrélation faible à nulle** entre la part de marché et l'OTP. Southwest, un des carriers les plus volumineux, maintient un OTP compétitif, tandis que de grands carriers legacy (American, United) peinent à convertir leur échelle en ponctualité. La discipline opérationnelle prime sur la taille.

### 5. Meilleur score composite
**Delta Air Lines** obtient le score composite le plus faible (meilleur) sur la décennie, suivi de près par **Alaska Airlines**. Ces deux compagnies se distinguent par une performance consistante sur les trois piliers : OTP, faible taux d'annulation et retards à l'arrivée maîtrisés.